# Fine-tune `sec-bert-base` on FiNER-139

Trains the model that `src/agents/financial_ner_agent.py` is written to load. Until this notebook has been run and its output pushed to the Hub, that agent has nothing to load and degrades to empty output by design.

**The task.** FiNER-139 tags *numeric tokens* in SEC filing text with 139 XBRL accounting concepts — deciding whether a given figure is `Revenues`, `NetIncomeLoss`, `OperatingLeaseLiability`, and so on. That is a different problem from the ORG/PER/LOC/MISC tagging `NERAgent` does, and much closer to what `KPIAgent` reaches for with eight regexes. `KPIAgent.REVENUE_XBRL_TAGS` is already wired to consume this model's output for the `revenue` KPI.

**Why this is the right second experiment.** The retrieval fine-tune ([recorded in the README](README.md)) improved FiQA by 2.3% NDCG@10 and *cost* 11.6% Hit@1 on filings — it learned forum-discussion language and this pipeline reads SEC prose. The lesson was that genre match matters. FiNER-139 is SEC filing text evaluated on SEC filing text, so this run is in-domain by construction. Different task type too — token classification rather than contrastive retrieval.

**Why `sec-bert-base` and not `sec-bert-num` / `sec-bert-shape`.** The sibling models replace numeric tokens with pseudo-tokens specifically to make this task easier, and the paper reports better numbers for them. This uses the plain base model deliberately: the point is a clean comparison against the paper's `sec-bert-base` row (**89.2% micro-F1**), not the best achievable score.

**Verified before writing this** (with `datasets` 3.6.0): the dataset loads, has **279 labels** (139 concepts × B-/I-, plus `O`), and splits of **900,384 train / 112,494 validation / 108,378 test**.

**Runtime:** Runtime → Change runtime type → **T4 GPU**. The default 50k subset takes roughly 40–60 minutes for 2 epochs.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"
print("\n", torch.cuda.get_device_name(0))

## 2. Install

`datasets` is pinned below 4.0 deliberately, and this is not a style choice. `nlpaueb/finer-139` is a legacy loading-script dataset — it ships `finer-139.py` and a zip, with no Parquet — and `datasets>=4.0` removed script execution entirely (`RuntimeError: Dataset scripts are no longer supported`). `trust_remote_code=True` is also required within this range.

Parquet mirrors of FiNER-139 exist on the Hub, but they are third-party, lightly used, and at least one is explicitly deduplicated — which changes the splits and would break comparability with the paper's number. The pin is the honest option.

In [ ]:
!pip install -q "datasets>=2.19,<4.0" "transformers>=4.36" seqeval accelerate

import datasets, transformers
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)

## 3. Load FiNER-139

In [ ]:
from datasets import load_dataset

raw = load_dataset("nlpaueb/finer-139", trust_remote_code=True)
print(raw)

label_names = raw["train"].features["ner_tags"].feature.names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

print(f"\n{len(label_names)} labels (expected 279)")
print("e.g.", label_names[:5])

## 4. Subsample — and keep validation separate from test

The full train split is ~900k sentences, which is many hours on a T4. The default trades final F1 for a run that finishes inside one session.

**Validation is used during training, test only at the end.** Training below sets `load_best_model_at_end` with `metric_for_best_model="f1"`, so the eval split decides *which checkpoint is kept*. Pointing that at the test split would select a checkpoint using test performance and then report that same test score — an optimistically biased number, and a subtle enough mistake that it usually goes unnoticed. FiNER-139 ships a real validation split; this uses it.

Report the subset sizes next to whatever F1 you get. A 50k-sample number is not comparable to the paper's full-split 89.2%.

In [ ]:
TRAIN_SAMPLES = 50_000   # None for the full 900,384
VAL_SAMPLES = 5_000      # used for checkpoint selection during training
TEST_SAMPLES = 5_000     # touched only once, at the end
SEED = 42

train_ds = raw["train"].shuffle(seed=SEED)
val_ds = raw["validation"]
test_ds = raw["test"]

if TRAIN_SAMPLES:
    train_ds = train_ds.select(range(min(TRAIN_SAMPLES, len(train_ds))))
if VAL_SAMPLES:
    val_ds = val_ds.select(range(min(VAL_SAMPLES, len(val_ds))))
if TEST_SAMPLES:
    test_ds = test_ds.select(range(min(TEST_SAMPLES, len(test_ds))))

print(f"train {len(train_ds):,}   validation {len(val_ds):,}   test {len(test_ds):,}")

## 5. Tokenize and align labels

The dataset is pre-tokenized into words, so `is_split_into_words=True`. WordPiece then splits words into subwords and the word-level tags have to be spread across them: the first subword carries the tag, continuation subwords get `-100` so the loss ignores them, and so do `[CLS]`/`[SEP]`/padding.

Getting this wrong is the classic silent failure in token classification — the model trains, the loss falls, and F1 is quietly meaningless because labels are offset from the tokens they describe.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "nlpaueb/sec-bert-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def tokenize_and_align(batch):
    encoded = tokenizer(
        batch["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,
    )

    all_labels = []
    for i, tags in enumerate(batch["ner_tags"]):
        word_ids = encoded.word_ids(batch_index=i)
        previous_word = None
        labels = []
        for word_id in word_ids:
            if word_id is None:
                labels.append(-100)             # special token
            elif word_id != previous_word:
                labels.append(tags[word_id])    # first subword of a word
            else:
                labels.append(-100)             # continuation subword
            previous_word = word_id
        all_labels.append(labels)

    encoded["labels"] = all_labels
    return encoded


train_tok = train_ds.map(tokenize_and_align, batched=True, remove_columns=train_ds.column_names)
val_tok = val_ds.map(tokenize_and_align, batched=True, remove_columns=val_ds.column_names)
test_tok = test_ds.map(tokenize_and_align, batched=True, remove_columns=test_ds.column_names)
print(train_tok)

## 6. Metrics

`seqeval` scores whole entity spans, not individual tokens — a prediction counts only if the entire span and its type are right. Token-level accuracy would look excellent purely from the `O` class, which overwhelmingly dominates this dataset.

In [ ]:
import numpy as np
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score


def decode(predictions, labels):
    """Drop -100 positions and map ids back to label strings."""
    true_labels, true_preds = [], []
    for prediction, label in zip(predictions, labels):
        kept = [(p, l) for p, l in zip(prediction, label) if l != -100]
        true_preds.append([id2label[p] for p, _ in kept])
        true_labels.append([id2label[l] for _, l in kept])
    return true_labels, true_preds


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    true_labels, true_preds = decode(np.argmax(logits, axis=-1), labels)
    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

## 7. Train

`eval_dataset` is the **validation** split. The best checkpoint is chosen on it, leaving the test split genuinely untouched until step 8.

In [ ]:
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

args = TrainingArguments(
    output_dir="sec-bert-finer139",
    learning_rate=3e-5,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,          # validation, not test - see step 4
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

## 8. Final evaluation on the held-out test split

The number to report. Quote it with the subset sizes attached — the paper's 89.2% is `sec-bert-base` trained on the full 900k split, so a 50k-sample model should land below it, and saying so is the difference between a result and a claim.

In [ ]:
test_metrics = trainer.evaluate(eval_dataset=test_tok, metric_key_prefix="test")

print(test_metrics)
print(f"\ntrained on {len(train_ds):,} | validated on {len(val_ds):,} | tested on {len(test_ds):,}")
print(f"test micro-F1: {test_metrics['test_f1']:.4f}")
print(f"  precision:   {test_metrics['test_precision']:.4f}")
print(f"  recall:      {test_metrics['test_recall']:.4f}")
print("\nreference: FiNER-139 paper, sec-bert-base, FULL train split: 0.892")

In [ ]:
# Per-concept breakdown: which of the 139 XBRL tags the model actually
# learned, versus which are simply too rare in the subset to learn at all.
predictions, labels, _ = trainer.predict(test_tok)
true_labels, true_preds = decode(np.argmax(predictions, axis=-1), labels)

print(classification_report(true_labels, true_preds, digits=3))

## 9. Push to the Hub

`FinancialNERAgent` loads the model by name, so this is the step that connects the training run to the pipeline.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
HF_REPO = "dataisgod/sec-bert-finer139"   # <- edit if you want a different name

trainer.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)

print(f"https://huggingface.co/{HF_REPO}")

## 10. Check it loads through the agent

Confirms the trained model is reachable via the pipeline's own code path, not just inside this notebook.

`FinancialNERAgent` currently lives on the `feature/financial-ner` branch, which is **9 commits behind `main`** and still uses the old full-state return contract. Wiring it into `main`'s parallel graph needs that updated first — see the note at the bottom.

In [ ]:
!git clone -q https://github.com/abhinaba01/fundamental-financial-analysis.git repo_check
%cd repo_check
!git checkout -q feature/financial-ner
!pip install -q transformers

from src.agents.financial_ner_agent import FinancialNERAgent

agent = FinancialNERAgent(model_name=HF_REPO, device="cuda")
print("model loaded:", agent.pipeline is not None)

sample = (
    "Total net sales were $383,285 million in fiscal 2023, "
    "and net income was $96,995 million."
)
for entity in agent.tag_texts([sample])[0]:
    print(f"  {entity['tag']:<50} {entity['text']!r}  score={entity['score']:.3f}")

%cd ..

## Notes — what to do with the result

**Report it against the right baseline.** The paper's 89.2% is the full 900k split; a 50k run is a different experiment. State both numbers and the subset size. If you want the comparable figure, set `TRAIN_SAMPLES = None` and expect many hours.

**Read the per-concept report, not just the micro-F1.** FiNER-139's tag distribution is severely skewed — a handful of concepts like `Revenues` dominate while dozens appear a few times. Micro-F1 hides that. "Strong on the 20 common concepts, near zero on the long tail" is the honest summary, and it is also the argument for why this helps `KPIAgent`: revenue is exactly the head of that distribution.

**Wiring it into the pipeline is a separate piece of work.** `feature/financial-ner` predates the parallel fan-out. Its `__call__` returns the whole state, which the parallel graph rejects — concurrent branches must return only their own keys, or LangGraph sees multiple writes to every key in one superstep. Bringing it onto `main` means rebasing, converting `__call__` to a delta return, and adding `financial_ner` as a fourth parallel node alongside `ner`, `sentiment` and `kpi`. Worth doing once the model exists and the F1 justifies it.

**A weak result is still worth recording.** If the long tail is unlearnable at this subset size, say so — that is a real finding about the dataset, and it sits naturally next to the retrieval negative result already in the README.